# PROGRAMMING IN JAVA — ASSIGNMENT 3 (CO3)
## Exception Handling and Multithreading

**Student:** Archana V  
**Register Number:** 192521420  
**Faculty:** Ravi Pujitha  
**Academic Year:** 2026–2027  
**Course Outcome:** CO3  
**Assignment Marks:** 50  

> Based on the uploaded faculty assignment PDF. The source contains **Q1–Q7**; this notebook separates the implementation and experiment components into **12 notebook tasks** for easier execution and review. The original question names and requirements are retained.


### Google Colab setup
Java source files will be written to the current working directory and compiled with `javac`, then executed with `java`. Each program is independent so that it can be rerun safely.

In [1]:
# Check the Java runtime (Colab normally has Java; install if missing)
import subprocess, shutil
if shutil.which("javac") is None:
    subprocess.run(["bash", "-lc", "apt-get update -qq && apt-get install -y openjdk-17-jdk"], check=True)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)
print(subprocess.run(["javac", "-version"], capture_output=True, text=True).stderr)

openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)




## Task 1 — Q1: Handling Failure in a Bank ATM Simulation
**Original topic:** Exception hierarchy, checked/unchecked exceptions, try, catch (including multi-catch), finally, input validation.


### Objective and failure analysis
- Account number input may be non-numeric (`InputMismatchException`, unchecked).
- Withdrawal amount may be non-numeric (`InputMismatchException`, unchecked), negative/zero (`IllegalArgumentException`, unchecked), or exceed the balance (`InsufficientFundsException`, user-defined unchecked exception in this demonstration).
- Card-reader file missing (`FileNotFoundException`, checked).
- `Error` subclasses represent serious JVM/system problems and should not normally be caught for ordinary ATM recovery.

**Relevant hierarchy (simplified):**
```mermaid
classDiagram
Throwable <|-- Error
Throwable <|-- Exception
Exception <|-- RuntimeException
RuntimeException <|-- IllegalArgumentException
RuntimeException <|-- InputMismatchException
Exception <|-- IOException
IOException <|-- FileNotFoundException
RuntimeException <|-- InsufficientFundsException
```

Catch blocks are ordered from specific to general. `finally` runs whether the try block succeeds or a handled exception occurs; here it reports completion of the transaction attempt.

In [2]:
%%writefile ATM.java
import java.util.*;
import java.io.*;
class InsufficientFundsException extends RuntimeException {
    InsufficientFundsException(String message) { super(message); }
}
public class ATM {
    static void withdraw(double balance, double amount) {
        try {
            if (amount <= 0) throw new IllegalArgumentException("Amount must be positive");
            if (amount > balance) throw new InsufficientFundsException("Insufficient balance");
            System.out.printf("Withdrawal successful. Remaining balance: %.2f%n", balance - amount);
        } catch (InputMismatchException | NumberFormatException e) {
            System.out.println("Input error: enter numeric values.");
        } catch (IllegalArgumentException e) {
            System.out.println("Invalid amount: " + e.getMessage());
        } catch (InsufficientFundsException e) {
            System.out.println("Transaction declined: " + e.getMessage());
        } finally {
            System.out.println("ATM transaction attempt completed.");
        }
    }
    public static void main(String[] args) {
        Scanner sc = new Scanner(System.in);
        try {
            System.out.print("Account number: "); String account = sc.next();
            System.out.print("Balance: "); double balance = sc.nextDouble();
            System.out.print("Withdrawal amount: "); double amount = sc.nextDouble();
            System.out.println("Processing account " + account);
            withdraw(balance, amount);
        } catch (InputMismatchException e) {
            System.out.println("Input error: enter numbers for balance and amount.");
        } finally { sc.close(); }
    }
}

Writing ATM.java


In [3]:
!javac ATM.java
!echo -e "A1001\n1000\n250" | java ATM

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Account number: Balance: Withdrawal amount: Processing account A1001
Withdrawal successful. Remaining balance: 750.00
ATM transaction attempt completed.


### Q1 tests and trace table
| Input scenario | Expected path | `finally` executes? |
|---|---|---|
| Balance 1000, amount 250 | Success | Yes |
| Balance 1000, amount 1200 | Insufficient funds | Yes |
| Amount -5 | IllegalArgumentException | Yes |
| Letters for numeric field | InputMismatchException | Yes (outer input handling) |
| Missing card file (if file access is added) | FileNotFoundException | Yes, if inside transaction try/finally |

**Stack trace interpretation:** the exception class and message appear first, followed by method calls in the stack (the top is where it was thrown, subsequent frames show callers). A broad `catch (Exception e)` can be acceptable at a program boundary for logging and controlled recovery, but it hides the distinction among failure types if used everywhere. It also does not catch `Error`.

**Alternative (validation + return codes):** check tokens with `Scanner.hasNextDouble()` / `hasNextInt()` before reading, and return status codes (e.g., `0=success`, `-1=invalid input`, `-2=insufficient balance`). This can avoid exception cost for expected invalid input but may require more branching and can be less expressive. Normal path: both approaches O(1) time and O(1) extra space. Exception creation/throwing has higher constant cost and captures stack-trace data; return-code validation has lower constant overhead for expected failures.

## Task 2 — Q2: User-Defined Exceptions and Propagation in an Online Examination Portal

**Original topic:** checked exceptions, `throw`, `throws`, propagation across controller/service/storage, exception chaining, `getCause()`.

### Exception design
```mermaid
classDiagram
Exception <|-- ExamException
ExamException <|-- SubmissionStorageException
ExamException <|-- ExamWindowClosedException
ExamException <|-- NotRegisteredException
ExamException <|-- SubmissionTooLargeException
```
The portal's domain exceptions are checked (`extends Exception`) because callers must explicitly handle expected business/storage failures. A storage-layer `IOException` is wrapped in a meaningful `SubmissionStorageException`, preserving the cause.

In [4]:
%%writefile PortalExceptions.java
import java.io.*;
class ExamException extends Exception {
    ExamException(String message) { super(message); }
    ExamException(String message, Throwable cause) { super(message, cause); }
}
class SubmissionStorageException extends ExamException {
    SubmissionStorageException(String message, Throwable cause) { super(message, cause); }
}
class ExamWindowClosedException extends ExamException { ExamWindowClosedException(String m){super(m);} }
class NotRegisteredException extends ExamException { NotRegisteredException(String m){super(m);} }
class SubmissionTooLargeException extends ExamException { SubmissionTooLargeException(String m){super(m);} }
class Storage {
    void save(String fileName, long sizeKb) throws SubmissionStorageException {
        try {
            if (sizeKb > 500) throw new IOException("Disk quota exceeded");
            System.out.println("Saved " + fileName);
        } catch (IOException e) {
            // TODO completed: preserve the IOException as the cause
            throw new SubmissionStorageException("Could not store " + fileName, e);
        }
    }
}
public class PortalExceptions {
    static void serviceSave(Storage storage, String file, long size) throws ExamException {
        storage.save(file, size); // propagation through service
    }
    static void controllerSave(Storage storage, String file, long size) {
        try { serviceSave(storage, file, size); }
        catch (ExamException e) {
            System.out.println("Error: " + e.getMessage());
            System.out.println("Cause: " + e.getCause().getMessage());
        }
    }
    public static void main(String[] args) {
        Storage s = new Storage();
        controllerSave(s, "answer01.pdf", 120);
        controllerSave(s, "answer02.pdf", 800);
    }
}

Writing PortalExceptions.java


In [5]:
!javac PortalExceptions.java
!java PortalExceptions

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Saved answer01.pdf
Error: Could not store answer02.pdf
Cause: Disk quota exceeded


### Q2 scenarios, chaining and alternative
| Scenario | Domain exception | Layer raised | Controller response |
|---|---|---|---|
| Window expired | `ExamWindowClosedException` | Service | Explain window is closed |
| Student not registered | `NotRegisteredException` | Service | Explain registration requirement |
| File too large | `SubmissionTooLargeException` | Service | Explain file-size limit |
| Storage quota error | `SubmissionStorageException` | Storage, propagated | Friendly storage message; cause retained |

**Propagation path:** Storage → Service → Controller → user-facing handler. If the cause is not passed to the wrapper constructor, `getCause()` is `null`, and the original low-level reason and stack context are lost. `throws` declares a possible exception; `throw` actually raises it.

**Unchecked alternative:** domain exceptions could extend `RuntimeException`; method signatures no longer need `throws`, and callers are not forced to catch. This is convenient but weakens compile-time handling guarantees. Exception creation/stack capture is roughly O(depth) in stack frames time and memory; chaining adds another exception reference and its stack data, so memory is O(sum of stack depths), typically bounded by stack depth in the program.

## Task 3 — Q3: Resource Safety, Debugging and Robust Code (manual close)

### Design
A report generator reads several input files and writes a summary. Manual closing in a `finally` block must close every opened resource, even if processing fails. A common bug is closing only the last successfully opened reader or allowing a close call to mask the primary exception. The example intentionally uses try/catch/finally and logs the input filename on errors.

In [6]:
%%writefile ManualReport.java
import java.io.*;
import java.util.*;
public class ManualReport {
    static int openReaders = 0;
    static int process(List<String> paths, String output) {
        int total = 0; BufferedWriter writer = null;
        try {
            writer = new BufferedWriter(new FileWriter(output));
            for (String path : paths) {
                BufferedReader reader = null;
                try {
                    reader = new BufferedReader(new FileReader(path)); openReaders++;
                    String line; while ((line = reader.readLine()) != null) total += Integer.parseInt(line.trim());
                } catch (IOException | NumberFormatException e) {
                    System.err.println("Failed while reading " + path + ": " + e.getMessage());
                } finally {
                    if (reader != null) try { reader.close(); openReaders--; } catch (IOException e) { System.err.println("Close failed: " + e); }
                }
            }
            writer.write("TOTAL=" + total); writer.newLine();
        } catch (IOException e) { System.err.println("Output error: " + e.getMessage()); }
        finally { if (writer != null) try { writer.close(); } catch (IOException e) { System.err.println("Writer close failed: " + e); } }
        return total;
    }
    public static void main(String[] args) throws Exception {
        try (PrintWriter p = new PrintWriter("store1.txt")) { p.println("10"); p.println("20"); }
        try (PrintWriter p = new PrintWriter("store2.txt")) { p.println("5"); p.println("15"); }
        System.out.println("Manual total: " + process(Arrays.asList("store1.txt", "missing.txt", "store2.txt"), "manual-summary.txt"));
        System.out.println("Tracked open readers after run: " + openReaders);
    }
}

Writing ManualReport.java


In [7]:
!javac ManualReport.java
!java ManualReport
!cat manual-summary.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Failed while reading missing.txt: missing.txt (No such file or directory)
Manual total: 50
Tracked open readers after run: 0
TOTAL=50


### Q3 resource tests and analysis
Test the same two valid store files with and without a missing or corrupt file. For this example, readers are closed in the per-file `finally`; the output writer is closed in the outer `finally`. A failure is logged with the file path, and processing continues for other files.

| Scenario | Expected behaviour | Reader handles remaining after run |
|---|---|---:|
| All files valid | Totals written | 0 |
| One file missing | Error logged; later files still processed | 0 |
| One line is not a number | Parse error logged; file still closed | 0 |

**Debugging suggestion:** set a breakpoint after reader creation, then step to the catch/finally on a deliberately missing/corrupt input. Track each open and close. Manual closing is fragile if close logic is omitted or if multiple resources are acquired. Complexity for `f` files with `L` total lines: O(L) time; manual and automatic versions require O(1) auxiliary memory when processing sequentially; manual implementation may leak file descriptors if cleanup is missed. Maximum simultaneously open file handles here is one input reader plus the output writer (2).

**Try-with-resources alternative:** `try (BufferedReader reader = new BufferedReader(new FileReader(path)))` closes automatically. Multiple resources close in reverse declaration order. If body and close both fail, close failure can be recorded as a suppressed exception. Try-with-resources does not fix bad input validation, incomplete output strategy, or incorrect transaction handling by itself.

## Task 4 — Q3: Re-implementation using try-with-resources

This version manages each reader and the writer automatically. It processes input files one at a time, so its resource use remains bounded.

In [8]:
%%writefile SafeReport.java
import java.io.*;
import java.util.*;
public class SafeReport {
    static int run(List<String> files, String output) {
        int sum = 0;
        try (BufferedWriter writer = new BufferedWriter(new FileWriter(output))) {
            for (String file : files) {
                try (BufferedReader reader = new BufferedReader(new FileReader(file))) {
                    String line;
                    while ((line = reader.readLine()) != null) sum += Integer.parseInt(line.trim());
                } catch (IOException | NumberFormatException e) {
                    System.err.println("Skipping " + file + " due to: " + e.getMessage());
                }
            }
            writer.write("TOTAL=" + sum); writer.newLine();
        } catch (IOException e) { System.err.println("Report output failure: " + e.getMessage()); }
        return sum;
    }
    public static void main(String[] args) throws Exception {
        try (PrintWriter p = new PrintWriter("store1.txt")) { p.println("10"); p.println("20"); }
        try (PrintWriter p = new PrintWriter("store2.txt")) { p.println("5"); p.println("15"); }
        System.out.println("Safe total: " + run(Arrays.asList("store1.txt", "missing.txt", "store2.txt"), "safe-summary.txt"));
    }
}

Writing SafeReport.java


In [9]:
!javac SafeReport.java
!java SafeReport
!cat safe-summary.txt

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Skipping missing.txt due to: missing.txt (No such file or directory)
Safe total: 50
TOTAL=50


### Compare resource strategies
| Strategy | Closes on normal path | Closes on exception | Leak risk | Handles (sequential) |
|---|---|---|---|---|
| Explicit close only | Yes | Not guaranteed | High | Depends on failure |
| `finally` close | Yes | Yes if cleanup correctly written | Moderate (cleanup bugs possible) | ≤2 in this design |
| try-with-resources | Yes | Yes automatically | Lowest | ≤2 in this design |

Both approaches process `f` files and `L` total records in O(L) time. Sequential reading holds O(1) extra memory. For resource robustness, prefer try-with-resources.

## Task 5 — Q4: Thread Creation and the Thread Life Cycle (extends Thread)

### Thread states
Java's `Thread.State` includes NEW, RUNNABLE, BLOCKED, WAITING, TIMED_WAITING, and TERMINATED. A station can move from NEW → RUNNABLE → TIMED_WAITING (during sleep) → RUNNABLE → TERMINATED. `start()` asks the JVM to schedule a new thread; calling `run()` directly executes on the current thread.

```mermaid
stateDiagram-v2
[*] --> NEW
NEW --> RUNNABLE: start()
RUNNABLE --> TIMED_WAITING: sleep()
TIMED_WAITING --> RUNNABLE: timeout
RUNNABLE --> BLOCKED: monitor lock unavailable
BLOCKED --> RUNNABLE: lock acquired
RUNNABLE --> WAITING: wait()/join()
WAITING --> RUNNABLE: notify()/joined thread completes
RUNNABLE --> TERMINATED: run() ends
TERMINATED --> [*]
```

In [10]:
%%writefile ThreadStations.java
class StationThread extends Thread {
    private final int station, scripts;
    StationThread(int station, int scripts) { this.station=station; this.scripts=scripts; }
    public void run() {
        try { Thread.sleep(30 + station * 15); }
        catch (InterruptedException e) { Thread.currentThread().interrupt(); }
        System.out.println("Station " + station + " scanned " + scripts + " scripts.");
    }
}
public class ThreadStations {
    public static void main(String[] args) throws InterruptedException {
        StationThread[] stations = new StationThread[4];
        for (int i=0; i<4; i++) { stations[i]=new StationThread(i+1, 25); stations[i].start(); }
        for (StationThread t: stations) t.join();
        System.out.println("All stations completed; total scripts = 100");
    }
}

Writing ThreadStations.java


In [11]:
!javac ThreadStations.java
!java ThreadStations

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Station 1 scanned 25 scripts.
Station 2 scanned 25 scripts.
Station 3 scanned 25 scripts.
Station 4 scanned 25 scripts.
All stations completed; total scripts = 100


### Thread method notes and tests
- `start()` creates a separate execution path; `run()` is just a method call if invoked directly.
- `sleep(ms)` pauses the current thread temporarily; it does not release held monitor locks.
- `join()` waits for the target thread to finish; the final total is printed after all four joins.
- `isAlive()` can be used to check whether a started thread has not yet terminated.
- Priorities are hints to the scheduler; they do not guarantee finish order.
- A daemon thread does not keep the JVM alive after all non-daemon threads end; use daemon status for background support tasks, not critical work.

Output order can vary due to scheduling. With 4 stations and 25 scripts each, the count is 100. For n scripts split into p workers, idealized time is approximately O(n/p) work per worker (plus scheduling and imbalance); creating one thread per station has stack overhead proportional to number of threads, while a pool caps worker threads.

## Task 6 — Q4: Thread Creation and the Thread Life Cycle (implements Runnable)

`Runnable` separates the job from the thread object, allowing the same task to be submitted to different thread-management mechanisms, including executor pools. This is generally preferred when the class already needs to extend another class or when tasks are managed by an executor.

In [12]:
%%writefile RunnableStations.java
class ScanJob implements Runnable {
    private final int station, scripts;
    ScanJob(int station, int scripts) { this.station=station; this.scripts=scripts; }
    public void run() {
        try { Thread.sleep(20 + station * 10); }
        catch (InterruptedException e) { Thread.currentThread().interrupt(); }
        System.out.println("Runnable station " + station + " finished " + scripts + " scripts.");
    }
}
public class RunnableStations {
    public static void main(String[] args) throws InterruptedException {
        Thread[] workers = new Thread[4];
        for (int i=0;i<4;i++) { workers[i]=new Thread(new ScanJob(i+1, 25)); workers[i].start(); }
        for (Thread worker: workers) worker.join();
        System.out.println("Runnable stations total = 100");
    }
}

Writing RunnableStations.java


In [13]:
!javac RunnableStations.java
!java RunnableStations

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Runnable station 2 finished 25 scripts.
Runnable station 3 finished 25 scripts.
Runnable station 4 finished 25 scripts.
Runnable station 1 finished 25 scripts.
Runnable stations total = 100


### Experiment table (record 10 runs)
The notebook code is deterministic in the total count but not in completion order. Run the cell repeatedly and record observed order; do not assume a particular order is guaranteed.
| Run | Finish order | Priority experiment | Daemon experiment |
|---|---|---|---|
| 1–10 | Fill from observed output | Record whether it changed (no guarantee) | Record whether background output completes before JVM exit |

**Thread vs Runnable:** extending `Thread` combines task and thread representation; implementing `Runnable` represents the work and can be passed to a `Thread` or executor. Each station processes a fixed batch. With n total scripts and p workers, ideal parallel workload is O(n/p), while total work remains O(n); memory includes per-worker stack plus task state.

## Task 7 — Q5: Race Conditions in a Cinema Seat-Booking System (unsynchronized)

A race condition occurs when threads interleave a check and update without atomicity. Two threads can both observe a seat as free, then both mark it booked and increment the booking count.

**Interleaving example:** T1 reads `false` → T2 reads `false` → T1 writes `true` and increments → T2 writes `true` and increments. One seat may be counted twice.

In [14]:
%%writefile UnsafeBooking.java
class UnsafeBooking {
    private final boolean[] booked;
    int bookingsMade=0;
    UnsafeBooking(int seats){booked=new boolean[seats];}
    void book(int seat){
        if(!booked[seat]) { // critical section is not protected
            Thread.yield();
            booked[seat]=true; bookingsMade++;
        }
    }
    public static void main(String[] args)throws InterruptedException{
        UnsafeBooking screen=new UnsafeBooking(10); Thread[] users=new Thread[100];
        for(int i=0;i<users.length;i++){final int seat=i%10;users[i]=new Thread(()->screen.book(seat));users[i].start();}
        for(Thread t:users)t.join(); System.out.println("Seats sold (unsafe): "+screen.bookingsMade);
    }
}

Writing UnsafeBooking.java


In [15]:
!javac UnsafeBooking.java
!java UnsafeBooking

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Seats sold (unsafe): 10


### Unsynchronized test
The unsynchronized program is intentionally unsafe. It may show more than 10 booking increments, but the result is timing-dependent; a run that happens to print 10 is not proof of correctness. The inserted `yield()` widens the race window but still cannot guarantee a duplicate in every run. Repeat the cell 10 times and record observed values.

Complexity per attempted booking is O(1); 100 attempts require O(100) work and the seat array uses O(s) space for s seats. Race effects can make the logical count exceed s.

## Task 8 — Q5: Race Conditions in a Cinema Seat-Booking System (synchronized)

The check-and-book operation is atomic when protected by the same intrinsic monitor. A synchronized method locks the `this` object; a synchronized block can lock only the critical section using that same object.

In [16]:
%%writefile SeatBooking.java
public class SeatBooking {
    private final boolean[] booked;
    private int bookingsMade=0;
    SeatBooking(int seats){booked=new boolean[seats];}
    public synchronized boolean book(int seat){
        if(booked[seat]) return false;
        booked[seat]=true; bookingsMade++; return true;
    }
    public static void main(String[] args)throws InterruptedException{
        SeatBooking screen=new SeatBooking(10); Thread[] users=new Thread[100];
        for(int i=0;i<users.length;i++){final int seat=i%10;users[i]=new Thread(()->screen.book(seat));users[i].start();}
        for(Thread t:users)t.join(); System.out.println("Seats sold: "+screen.bookingsMade);
    }
}

Writing SeatBooking.java


In [17]:
!javac SeatBooking.java
!java SeatBooking

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Seats sold: 10


### Completed TODO and verification
Expected output is `Seats sold: 10`. The synchronized method ensures only one thread at a time can execute the check-then-book logic on this booking object. A synchronized block around just the critical section can use `synchronized(this)` and the same intrinsic lock; locking only that section may reduce lock duration but is easier to misuse.
- Static synchronization locks the `Class` object and is useful when shared static state must be guarded across instances.
- Alternatives: `AtomicBoolean[]` with `compareAndSet(false,true)` gives lock-free seat claim; `ReentrantLock` gives explicit lock/unlock and advanced options; `ConcurrentHashMap.putIfAbsent` works for seat IDs represented as map keys.
- Per-seat atomic state needs O(s) space for s seats; each booking attempt is expected O(1). Under contention, retries/lock waiting may increase latency.
- For a multiplex, protect seat state per screen/seat or use database transaction constraints; one global lock may serialize unrelated bookings.

## Task 9 — Q6: Deadlock in Hostel Wallet Transfers and Prevention

Deadlock requires four conditions: mutual exclusion, hold-and-wait, no preemption, and circular wait. If account A holds its lock while waiting for B, and B holds its lock while waiting for A, both can remain blocked. Prevention uses a consistent global lock order (for example, lock the account with the lower ID first).

In [18]:
%%writefile TransferDemo.java
class Wallet { final int id; int balance; Wallet(int id,int b){this.id=id;balance=b;} }
public class TransferDemo {
    static void transferSafe(Wallet a, Wallet b, int amount) {
        Wallet first=a.id<b.id?a:b, second=a.id<b.id?b:a;
        synchronized(first){ synchronized(second){ if(a.balance>=amount){a.balance-=amount;b.balance+=amount;} } }
    }
    public static void main(String[] args)throws InterruptedException{
        Wallet a=new Wallet(1,1000), b=new Wallet(2,1000);
        Thread t1=new Thread(()->transferSafe(a,b,100));
        Thread t2=new Thread(()->transferSafe(b,a,50));
        t1.start();t2.start();t1.join();t2.join();
        System.out.println("Wallet balances: A="+a.balance+", B="+b.balance+", total="+(a.balance+b.balance));
    }
}

Writing TransferDemo.java


In [19]:
!javac TransferDemo.java
!java TransferDemo

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Wallet balances: A=950, B=1050, total=2000


### Safe-transfer verification and alternative
Total wallet funds remain 2000; after the two transfers, balances are A=950 and B=1050 (order is deterministic for these sequentially completed transfers). Lock ordering prevents circular wait because every thread requests locks in ascending account ID order.

Alternative: `tryLock(timeout)` attempts both locks, releases any acquired lock if the second fails, and retries or reports failure after timeout. This reduces indefinite waiting but can cause retries/starvation. Per-transfer work is O(1); locks add O(1) state. With concurrent transfers, ordered locks are straightforward and less error-prone than timeout/retry logic.

## Task 10 — Q6: Inter-Thread Communication (Producer–Consumer Token Counter)

The shared buffer has capacity 5. `put()` waits while full; `take()` waits while empty. Both conditions must be checked in `while` loops because a thread may wake spuriously or another thread may consume the condition before it reacquires the lock. `notifyAll()` wakes all waiters so the correct condition-checking thread can proceed.

`wait()` releases the monitor while waiting; `sleep()` pauses but does not release a monitor lock. Both `wait()` and `notifyAll()` must be called while holding the synchronized object's monitor.

In [20]:
%%writefile TokenCounter.java
public class TokenCounter {
    private final int[] buffer=new int[5];
    private int count=0,in=0,out=0;
    public synchronized void put(int token)throws InterruptedException{
        while(count==buffer.length) wait();
        buffer[in]=token; in=(in+1)%buffer.length; count++;
        notifyAll();
    }
    public synchronized int take()throws InterruptedException{
        while(count==0) wait();
        int token=buffer[out]; out=(out+1)%buffer.length; count--;
        notifyAll(); return token;
    }
    public static void main(String[] args)throws InterruptedException{
        TokenCounter counter=new TokenCounter(); int[] collected=new int[2];
        Thread cook=new Thread(()->{try{for(int t=1;t<=20;t++)counter.put(t);}catch(InterruptedException e){Thread.currentThread().interrupt();}});
        Thread[] students=new Thread[2];
        for(int s=0;s<2;s++){final int id=s;students[s]=new Thread(()->{try{for(int k=0;k<10;k++)collected[id]+=counter.take();}catch(InterruptedException e){Thread.currentThread().interrupt();}});}
        cook.start();for(Thread t:students)t.start();cook.join();for(Thread t:students)t.join();
        System.out.println("Sum of tokens collected: "+(collected[0]+collected[1])+" (expected 210)");
    }
}

Writing TokenCounter.java


In [21]:
!javac TokenCounter.java
!java TokenCounter

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Sum of tokens collected: 210 (expected 210)


### Token counter expected result
Twenty tokens numbered 1 through 20 are produced and consumed exactly once. Their sum is 210. The buffer count must always remain in the range 0–5.

| Test | Expected invariant |
|---|---|
| Producer faster than consumer | count ≤ 5; producer blocks at full |
| Consumer faster than producer | count ≥ 0; consumer blocks at empty |
| Two consumers | Each token consumed once; total sum 210 |
| Replace `while` with `if` | Potential incorrect access after spurious wake-up or competing wake-up; loops re-check predicate |

Alternative: `ArrayBlockingQueue<Integer>(5)` handles bounded blocking safely with `put` and `take`. Queue operation is O(1) expected and buffer space O(capacity). For account transfers, `tryLock(timeout)` is an alternative to ordered locks; ordered locks are generally simpler to reason about.

## Task 11 — Q7: Design a Concurrent, Fault-Tolerant Processor from First Principles

### Stage 1: requirements and interpretation
- Process 10,000 score files; independent files can be processed in parallel.
- Shared output: success/failure counts and result collection.
- A corrupt or missing file must not terminate all work.
- Each file is read, validated, and converted into a result.
- Completion summary must include every file outcome.

### Measurable requirements
1. Every input file receives exactly one outcome (success or failure).
2. One bad file does not stop remaining tasks.
3. Final success + failure count equals submitted file count.
4. Pool-size experiments use 1, 2, 4, and 8 threads and report measured elapsed time.

### Class design
```mermaid
classDiagram
class ResultProcessor { +process(Path): CallableResult }
class FileResult { +fileName + success + score + message }
class ProcessingSummary { +successCount + failureCount + results }
ResultProcessor --> FileResult
ProcessingSummary o-- FileResult
```

Use `ExecutorService`, `Callable`, and `Future`; use `ConcurrentLinkedQueue` to hold results safely. The main thread collects each Future and records a failure result if the task throws.

In [22]:
%%writefile ScoreProcessor.java
import java.nio.file.*;
import java.util.*;
import java.util.concurrent.*;
class FileResult {
    final String file; final boolean success; final int score; final String message;
    FileResult(String f, boolean ok, int s, String m){file=f;success=ok;score=s;message=m;}
}
public class ScoreProcessor {
    static FileResult process(Path file) {
        try {
            String text=Files.readString(file).trim();
            if(text.isEmpty()) throw new IllegalArgumentException("empty file");
            int score=Integer.parseInt(text);
            if(score<0||score>100) throw new IllegalArgumentException("score out of range");
            return new FileResult(file.toString(),true,score,"OK");
        } catch(Exception e) { return new FileResult(file.toString(),false,0,e.getClass().getSimpleName()+": "+e.getMessage()); }
    }
    static void runPool(int poolSize, List<Path> files) throws InterruptedException {
        ExecutorService pool=Executors.newFixedThreadPool(poolSize);
        ConcurrentLinkedQueue<FileResult> results=new ConcurrentLinkedQueue<>();
        List<Future<FileResult>> futures=new ArrayList<>();
        long start=System.nanoTime();
        for(Path f:files) futures.add(pool.submit(()->process(f)));
        for(int i=0;i<futures.size();i++) {
            try { results.add(futures.get(i).get()); }
            catch(ExecutionException e){results.add(new FileResult(files.get(i).toString(),false,0,"Task failed: "+e.getCause()));}
        }
        pool.shutdown(); pool.awaitTermination(1,TimeUnit.MINUTES);
        long elapsed=(System.nanoTime()-start)/1_000_000;
        int ok=0,fail=0; for(FileResult r:results){if(r.success)ok++;else fail++;}
        System.out.println("Threads="+poolSize+" Success="+ok+" Failure="+fail+" Total="+(ok+fail)+" Time(ms)="+elapsed);
    }
    public static void main(String[] args)throws Exception{
        List<Path> files=new ArrayList<>();
        for(int i=0;i<20;i++){Path p=Path.of("score"+i+".txt");Files.writeString(p,(i==4?"corrupt":String.valueOf((i*7)%101)));files.add(p);}
        files.add(Path.of("missing-score.txt"));
        for(int threads:new int[]{1,2,4,8}) runPool(threads,files);
    }
}

Writing ScoreProcessor.java


In [23]:
!javac ScoreProcessor.java
!java ScoreProcessor

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Threads=1 Success=19 Failure=2 Total=21 Time(ms)=36
Threads=2 Success=19 Failure=2 Total=21 Time(ms)=6
Threads=4 Success=19 Failure=2 Total=21 Time(ms)=6
Threads=8 Success=19 Failure=2 Total=21 Time(ms)=6


### Stage 3 experiment table
The program creates a small test batch (20 generated files, one deliberately corrupt and one missing) for safe demonstration. For each pool size it reports measured wall-clock milliseconds and outcome counts. Expected hand-check: 19 successful valid numeric files and 2 failures (corrupt + missing), so total 21. Timing is environment-dependent; do not treat one run as a benchmark conclusion. For a full assignment-scale test, change the generation loop to 10,000 files and record runs.

| Pool size | Successes | Failures | Total | Elapsed ms |
|---:|---:|---:|---:|---:|
| 1 | Output cell | Output cell | 21 | Output cell |
| 2 | Output cell | Output cell | 21 | Output cell |
| 4 | Output cell | Output cell | 21 | Output cell |
| 8 | Output cell | Output cell | 21 | Output cell |

**Why more threads may not help:** disk I/O, task overhead, CPU limits, memory, and contention may dominate. For independent files, ideal processing is roughly O(n/p) work per worker, but total work is O(n). Executor memory is O(n) for the Future/result lists in this sample plus O(p) worker stacks. Pool threads cap stack use compared with creating one thread per file.

## Task 12 — Q7: Alternatives, Evaluation and Big-O Comparison

### Alternatives
1. **Parallel stream:** `files.parallelStream().map(ScoreProcessor::process).collect(...)` uses a common ForkJoin pool by default. It is concise, but custom pool control, task-level failure reporting, and isolated per-file outcomes need deliberate design.
2. **One thread per file:** create a thread for every file. This is easy conceptually for a small number of files but impractical at 10,000 because thread creation, scheduling and stack memory are substantial.
3. **ExecutorService + Callable + Future:** fixed worker count, explicit results, and controlled shutdown; a good fit for this assignment.

### Big-O comparison
| Approach | Work/time idealization | Extra space | Failure handling |
|---|---|---|---|
| Sequential | O(n) | O(1) worker memory | Handle each file in loop |
| Fixed pool p | O(n/p) ideal, O(n) total | O(p) threads + O(n) queued/results | Per-task catch and Future outcomes |
| Parallel stream | O(n/p) ideal; overhead dependent | Pool-sized workers + collected results | Exceptions can propagate; handle inside map |
| Thread per file | O(n/p) ideal only if resources permit; practical overhead high | O(n) thread stacks | Individual try/catch needed |

### Submission checklist from faculty PDF
- [ ] Create repository `CSA09-192521420-Assignment3` (public or faculty collaborator added).
- [ ] Include a folder for each question/task, Java sources, input files, output screenshots.
- [ ] README: name, register number, Java version, compile/run steps and task result notes.
- [ ] Commit progressively (at least one commit per task).
- [ ] Add GitHub repository URL to report/cover page and submit the link.
- [ ] Include both students' individual contribution record (Student A / Student B / Joint), test tables, logs, and raw measurements.

### Individual contribution record (fill in after pair work)
| Question | Student A primary responsibility | Student B primary responsibility | Joint |
|---|---|---|---|
| Q1 | Exception design | Testing/tracing | ATM implementation & review |
| Q2 | Hierarchy/propagation design | Scenarios/chaining evidence | Portal design |
| Q3 | Debugging/measurement | Design/correctness | Resource implementations |
| Q4 | Thread design | Behavioural experiments | Thread comparison |
| Q5 | Race reproduction | Synchronization comparison | Booking implementation |
| Q6 | Producer-consumer design | Deadlock analysis | Both implementations |
| Q7 | Design | Implementation/measurement | Evaluation |

### Final reflection
The assignment demonstrates that reliable Java programs must handle failures deliberately, release resources, and coordinate concurrent operations safely. Checked exceptions make handling explicit, try-with-resources reduces cleanup errors, thread synchronization prevents races, and bounded executors plus per-task failure handling support resilient batch processing.